In [ ]:
from dotenv import load_dotenv
import os
from langchain.tools import tool

load_dotenv()


True

In [2]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

### 1.만든 TodoToolkit 가져오기

In [3]:
from todo_toolkit.toolkit import TodoToolkit

### 2.LLM 초기화

In [ ]:
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0
)

### 3.TodoToolkit 인스턴스 생성 및 도구 가져오기

In [5]:
todo_toolkit = TodoToolkit()
tools = todo_toolkit.get_tools()
print(tools)

[AddTodoTool(), ViewTodosTool(), CompleteTodoTool()]


### 4.프롬프트 설정
- 여기서는 간단한 예시를 사용합니다. 좀 더 정교한 프롬프트가 필요할 수 있습니다.

In [6]:
SYSTEM_PROMPT = """
You are a helpful assistant that can manage a todo list.
Use the provided tools to answer user requests.
Remember the conversation within the same thread.
"""


### 5.에이전트 생성
- `create_agent`는 LangChain v1의 표준 에이전트 생성 함수입니다.
- `model`, `tools`, `system_prompt`를 넘기면 내부적으로 LangGraph 기반 실행 흐름을 구성합니다.

In [7]:
checkpointer = InMemorySaver()
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer
)

config = {'configurable':{'thread_id':'user123'}}



### 6.에이전트 실행
- LangChain v1 에이전트는 `messages` 입력을 받아 실행합니다.
- 짧은 대화 기억은 `InMemorySaver` checkpointer와 `thread_id` 설정으로 관리합니다.
- 같은 `thread_id`로 호출하면 이전 대화 흐름을 이어서 사용할 수 있습니다.

In [8]:
# LangChain v1 방식의 short-term memory는 checkpointer + thread_id를 사용합니다.
# 별도 대화 기록 래퍼 없이 agent가 thread별 메시지를 저장합니다.


In [12]:
def run_todo_agent(user_input: str):
    result = agent.invoke(
        {'messages' : [{'role':'user', 'content':user_input}]},
        config=config
    )
    print(result['messages'][-1].content)
    return result

In [10]:
# 같은 thread_id("user123")로 호출하면 이전 대화와 도구 실행 결과가 이어집니다.
# 다른 사용자/세션은 config의 thread_id를 바꿔 분리합니다.


In [13]:
print("무엇을 도와드릴까요? (예: '점심 약속 추가해줘', '오늘 할 일 목록 보여줘')")
user_input = '점심 약속 추가해줘'
result = run_todo_agent(user_input)
result

무엇을 도와드릴까요? (예: '점심 약속 추가해줘', '오늘 할 일 목록 보여줘')
'점심 약속'이 할 일 목록에 추가되었습니다. 다른 할 일도 추가하시겠어요?


{'messages': [HumanMessage(content='점심 약속 추가해줘', additional_kwargs={}, response_metadata={}, id='b91f40db-30e5-4a0e-85d8-fa95d58ceef3'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 150, 'total_tokens': 168, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5efe265edd', 'id': 'chatcmpl-Dtks3awPQ874mFC4Q3YDzo3O4giUU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ef23d-948d-78f0-ae10-de3996d6b215-0', tool_calls=[{'name': 'add_todo', 'args': {'item': '점심 약속'}, 'id': 'call_af0AwBdh4pKfkqZQDRGLUBny', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 150, 'output_tokens': 18, 'tot

In [14]:
print("무엇을 도와드릴까요? (예: '점심 약속 추가해줘', '오늘 할 일 목록 보여줘')")
user_input = '저녁에 롯데월드 놀러 갈 거야'
result = run_todo_agent(user_input)
result

무엇을 도와드릴까요? (예: '점심 약속 추가해줘', '오늘 할 일 목록 보여줘')
'저녁에 롯데월드 놀러 가기'가 할 일 목록에 추가되었습니다. 더 추가할 할 일이 있나요?


{'messages': [HumanMessage(content='점심 약속 추가해줘', additional_kwargs={}, response_metadata={}, id='b91f40db-30e5-4a0e-85d8-fa95d58ceef3'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 150, 'total_tokens': 168, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5efe265edd', 'id': 'chatcmpl-Dtks3awPQ874mFC4Q3YDzo3O4giUU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ef23d-948d-78f0-ae10-de3996d6b215-0', tool_calls=[{'name': 'add_todo', 'args': {'item': '점심 약속'}, 'id': 'call_af0AwBdh4pKfkqZQDRGLUBny', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 150, 'output_tokens': 18, 'tot

In [15]:
print("무엇을 도와드릴까요? (예: '점심 약속 추가해줘', '오늘 할 일 목록 보여줘')")
user_input = '롯데월드 다녀왔어'
result = run_todo_agent(user_input)
result

무엇을 도와드릴까요? (예: '점심 약속 추가해줘', '오늘 할 일 목록 보여줘')
롯데월드 놀러 가는 할 일을 완료 처리했습니다. 다른 할 일도 완료하거나 추가할 것이 있나요?


{'messages': [HumanMessage(content='점심 약속 추가해줘', additional_kwargs={}, response_metadata={}, id='b91f40db-30e5-4a0e-85d8-fa95d58ceef3'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 150, 'total_tokens': 168, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5efe265edd', 'id': 'chatcmpl-Dtks3awPQ874mFC4Q3YDzo3O4giUU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ef23d-948d-78f0-ae10-de3996d6b215-0', tool_calls=[{'name': 'add_todo', 'args': {'item': '점심 약속'}, 'id': 'call_af0AwBdh4pKfkqZQDRGLUBny', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 150, 'output_tokens': 18, 'tot

In [16]:
print("무엇을 도와드릴까요? (예: '점심 약속 추가해줘', '오늘 할 일 목록 보여줘')")
user_input = '남아있는 일정 정리해줘'
result = run_todo_agent(user_input)
result

무엇을 도와드릴까요? (예: '점심 약속 추가해줘', '오늘 할 일 목록 보여줘')
현재 남아있는 일정은 '점심 약속' 하나입니다. 더 도와드릴 일이 있을까요?


{'messages': [HumanMessage(content='점심 약속 추가해줘', additional_kwargs={}, response_metadata={}, id='b91f40db-30e5-4a0e-85d8-fa95d58ceef3'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 150, 'total_tokens': 168, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5efe265edd', 'id': 'chatcmpl-Dtks3awPQ874mFC4Q3YDzo3O4giUU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ef23d-948d-78f0-ae10-de3996d6b215-0', tool_calls=[{'name': 'add_todo', 'args': {'item': '점심 약속'}, 'id': 'call_af0AwBdh4pKfkqZQDRGLUBny', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 150, 'output_tokens': 18, 'tot